(reactions)=
# Reactions

In FESTIM, chemical reactions between species are defined (in the bulk) with ``Reaction`` objects.

We will learn in this tutorial how these can be used a building blocks for many different problems.

**Objectives**

- Define simple chemical reactions
- Understand how trapping can be defined as a reaction
- Define more complex reaction scheme (multi-occupancy trapping, decay, etc.)

## Simple reaction

We'll start with a simple reaction between species A and B forming species C at a rate $k$ and assume there is no backward reaction.

```{math}
\ce{A + B ->[k] C}
```

The governing equations for this problem are:

\begin{align}
    \frac{\partial c_A}{\partial t} &= \nabla \cdot (D \nabla c_A) - k c_A c_B\\
    \frac{\partial c_B}{\partial t} &= \nabla \cdot (D \nabla c_A) -k c_A c_B\\
    \frac{\partial c_C}{\partial t} &= k c_A c_B
\end{align}

In [ ]:
import festim as F
import numpy as np

my_model = F.HydrogenTransportProblem()  # top level object, wraps all the mesh/function space/BC/solver stuff we were building by hand before


my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))  # same idea as our own create_mesh, just gives us the 1D interval mesh from 100 points between 0 and 1 without us having to build cells/mesh_points ourselves

left_surf = F.SurfaceSubdomain1D(id=1, x=0)  # these are the boundary markers - basically the facet tagging we did by hand (locate_entities_boundary + meshtags), festim just does it for us
right_surf = F.SurfaceSubdomain1D(id=2, x=1)

# assumes the same diffusivity for all species
material = F.Material(D_0=1, E_D=0)  # defines D as an arrhenius law, D = D_0*exp(-E_D/kT). setting E_D=0 just makes it a constant D_0 regardless of temperature

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)  # ties the material to the actual domain (0 to 1), so festim knows what diffusivity to use where

my_model.subdomains = [vol, left_surf, right_surf]  # registers all the subdomains (the volume + both surfaces) with the model

``Reaction`` object need a list of reactants, a list of product (we'll see later that it can be an empty list), as well as forward and backward reaction rates expressed as Arrhenius laws.

Let's define three ``Species`` object, noting that Species C is immobile.

In [ ]:
A = F.Species("A")  # mobile by default, so this diffuses (like our cm)
B = F.Species("B")  # also mobile by default
C = F.Species("C", mobile=False)  # this is the product, immobile so no diffusion term - same idea as our ct/c_hyd being DG
my_model.species = [A, B, C]

my_model.reactions = [
    F.Reaction(
        reactant=[A, B],  # A and B get consumed - exactly the reactant bookkeeping we did by hand with reaction/reaction_HO/reaction_hyd
        product=[C],       # C gains what A and B lose
        k_0=0.01,          # forward rate constant, arrhenius prefactor
        E_k=0,             # activation energy for k - 0 here so k is just a flat 0.01, no temperature dependence
        # reverse reaction with a rate of zero
        p_0=0,
        E_p=0,
        volume=vol,        # which subdomain this reaction happens in
    )
]

```{admonition} Tip
:class: tip
If your reaction rate isn't an Arrhenius law but say a constant value, simply set the activation energy (``E_k`` or ``E_p``) to 0.
```

We can then define boundary conditions for all the species, temperature, settings, and run the model:

In [ ]:
my_model.boundary_conditions = [
    # A BCs
    F.FixedConcentrationBC(left_surf, value=10, species=A),  # this is literally our fem.dirichletbc - value, which species, which surface, same (value, dofs, V) recipe just wrapped up
    F.FixedConcentrationBC(right_surf, value=0, species=A),
    # B BCs
    F.FixedConcentrationBC(left_surf, value=0, species=B),
    F.FixedConcentrationBC(right_surf, value=5, species=B),
]

my_model.temperature = 300  # still a required attribute even though E_D=0 here (D isn't actually temperature dependent in this example)

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=50)  # atol/rtol are the same snes_atol/snes_rtol we set by hand in petsc_options, final_time is our final_time

my_model.settings.stepsize = F.Stepsize(1)  # this is our dt

my_model.initialise()  # builds all the function spaces/forms/BCs under the hood - everything we did by hand with fem.functionspace, ufl.split, locate_dofs_topological etc
my_model.run()  # this is our whole "build F, wrap in NonlinearProblem, while loop calling solver.solve()" all done for us

In [ ]:
import matplotlib.pyplot as plt


def plot_profile(species, **kwargs):
    index = my_model.species.index(species)  # find which slot this species is in the mixed space - like our V.sub(0), V.sub(1)... but by lookup instead of us hardcoding it
    V0, dofs = my_model.function_space.sub(index).collapse()  # same collapse() trick we used for c1_pp/c2_pp - pulls this species into its own standalone space + gives the index map
    coords = V0.tabulate_dof_coordinates()[:, 0]  # same tabulate_dof_coordinates()[:,0] we used for plotting eg_1.py by hand
    sort_coords = np.argsort(coords)  # same sorting so the line plots left to right properly instead of jumping around
    c = my_model.u.x.array[dofs][sort_coords]  # pulling this species' values out of the big mixed solution vector - same idea as our map_c1_to_u indexing
    x = coords[sort_coords]
    return plt.plot(x, c, **kwargs)


# this is reused for every example below, so once plot_profile is built we just loop over species and plot each one
for species in my_model.species:
    plot_profile(species, label=species.name)

plt.xlabel("Position")
plt.ylabel("Concentration")
plt.legend()
plt.show()

As expected, the concentration of Species C is highest where the product $c_A c_B$ is highest.

## Two-way reaction

The previous example can be very easily extended to a two-way reaction:

```{math}
\ce{A + B <-->[k][p] C}
```

The governing equations for this problem are:

\begin{align}
    \frac{\partial c_A}{\partial t} &= \nabla \cdot (D \nabla c_A) - k c_A c_B + p c_C\\
    \frac{\partial c_B}{\partial t} &= \nabla \cdot (D \nabla c_B) -k c_A c_B+ p c_C\\
    \frac{\partial c_C}{\partial t} &= k c_A c_B - p c_C
\end{align}

In [ ]:
my_model = F.HydrogenTransportProblem()  # fresh model instance - not continuing the old one, this is a whole new problem

A = F.Species("A")
B = F.Species("B")
C = F.Species("C", mobile=False)
my_model.species = [A, B, C]

my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))

left_surf = F.SurfaceSubdomain1D(id=1, x=0)
right_surf = F.SurfaceSubdomain1D(id=2, x=1)

# assumes the same diffusivity for all species
material = F.Material(D_0=1, E_D=0)

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)

my_model.subdomains = [vol, left_surf, right_surf]

my_model.reactions = [
    F.Reaction(
        reactant=[A, B],
        product=[C],
        k_0=0.01,
        E_k=0,
        p_0=0.1,  # THIS is the only real change from the first example - now nonzero, so C can decay back into A+B. same as our reaction_hyd having a tiny nonzero p_hyd instead of a strict one-way sink
        E_p=0,
        volume=vol,
    )
]

my_model.boundary_conditions = [
    # A BCs
    F.FixedConcentrationBC(left_surf, value=10, species=A),
    F.FixedConcentrationBC(right_surf, value=0, species=A),
    # B BCs
    F.FixedConcentrationBC(left_surf, value=0, species=B),
    F.FixedConcentrationBC(right_surf, value=5, species=B),
]

my_model.temperature = 300

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=50)

my_model.settings.stepsize = F.Stepsize(1)

my_model.initialise()
my_model.run()

In [ ]:
# reusing plot_profile from the first example, works fine since it just reads whatever my_model currently points to
for species in my_model.species:
    plot_profile(species, label=species.name)

plt.xlabel("Position")
plt.ylabel("Concentration")
plt.legend()
plt.show()

## Chain reaction

Here we consider a reaction scheme where mobile species A and B react to form species C (immobile).
Species C then reacts *on its own* to form species D.

```{math}
\ce{A + B <-->[k_1][p_1] C}
```

```{math}
\ce{C ->[k_2] D}
```

The governing equations of this system are:

\begin{align}
    \frac{\partial c_A}{\partial t} &= \nabla \cdot (D \nabla c_A) - k_1 c_A c_B + p_1 c_C\\
    \frac{\partial c_B}{\partial t} &= \nabla \cdot (D \nabla c_B) -k_1 c_A c_B+ p_1 c_C\\
    \frac{\partial c_C}{\partial t} &= k_1 c_A c_B - p_1 c_C - k_2 c_C\\
    \frac{\partial c_D}{\partial t} &= \nabla \cdot (D \nabla c_D) + k_2 c_C\\
\end{align}

To implement this in a FESTIM model, one simply has to add both reactions with two ``Reaction`` objects.

In [ ]:
my_model = F.HydrogenTransportProblem()

A = F.Species("A")
B = F.Species("B")
C = F.Species("C", mobile=False)
D = F.Species("D")  # new species, mobile by default this time
my_model.species = [A, B, C, D]

my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))

left_surf = F.SurfaceSubdomain1D(id=1, x=0)
right_surf = F.SurfaceSubdomain1D(id=2, x=1)

# assumes the same diffusivity for all species
material = F.Material(D_0=1, E_D=0)

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)

my_model.subdomains = [vol, left_surf, right_surf]

my_model.reactions = [
    F.Reaction(
        reactant=[A, B],
        product=[C],  # same reversible A+B->C reaction as before
        k_0=0.01,
        E_k=0,
        p_0=0.1,
        E_p=0,
        volume=vol,
    ),
    F.Reaction(
        reactant=[C],   # THIS is the chain: C is a product above and a reactant here, so whatever C builds up from reaction 1 then feeds into reaction 2
        product=[D],
        k_0=0.2,
        E_k=0,
        p_0=0,  # one-way, like our reaction_hyd - C doesn't convert back from D
        E_p=0,
        volume=vol,
    ),
]

my_model.boundary_conditions = [
    # A BCs
    F.FixedConcentrationBC(left_surf, value=10, species=A),
    F.FixedConcentrationBC(right_surf, value=0, species=A),
    # B BCs
    F.FixedConcentrationBC(left_surf, value=0, species=B),
    F.FixedConcentrationBC(right_surf, value=5, species=B),
]
# note there's no BC for C or D at all - same as our ct/c_hyd having no BCs, they're purely reaction-driven

my_model.temperature = 300

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=50)

my_model.settings.stepsize = F.Stepsize(1)

my_model.initialise()
my_model.run()

In [ ]:
for species in my_model.species:
    plot_profile(species, label=species.name)

plt.xlabel("Position")
plt.ylabel("Concentration")
plt.legend()
plt.show()

After 5 s, we can see that the concentration of D is fairly flat. This is because D is mobile, but the boundary conditions for this species are no-flux conditions. Species D will therefore diffuse but won't "escape" the domain.

## Trapping


Hydrogen trapping can be represented as a reaction too. Here we consider mobile hydrogen ($\mathrm{H}$) reacting with an empty trap ($[\ ]$) to form trapped hydrogen ($\mathrm{[H]}$):

```{math}
\ce{H + [ ] <-->[k][p] [H]}
```


The governing equations for this problem are:

\begin{align}
    \frac{\partial c_\mathrm{m}}{\partial t} &= \nabla \cdot (D \nabla c_\mathrm{m}) - k c_\mathrm{m} n_\mathrm{empty} + p c_\mathrm{t}\\
    \frac{\partial n_\mathrm{empty}}{\partial t} &= -k c_\mathrm{m} n_\mathrm{empty} + p c_\mathrm{t}\\
    \frac{\partial c_\mathrm{t}}{\partial t} &= k c_\mathrm{m} n_\mathrm{empty} - p c_\mathrm{t}
\end{align}


Following what was done in the [Species](species.ipynb) tutorial, we will show that empty traps can be represented either as an explicit or implicit species.


```{note}
The [TDS simulation tutorial](../applications/task02.ipynb) provides a more complete and realistic example including temperature-dependent trapping/detrapping rates.
```

### Explicit trapping sites

Here, we treat the empty trapping sites as an explicit species with an initial concentration.

In [ ]:
my_model = F.HydrogenTransportProblem()
my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))

material = F.Material(D_0=1, E_D=0)

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)

mobile_H = F.Species("H")  # this is our cm
trapped_H = F.Species("H_trapped", mobile=False)  # this is our ct
empty_traps = F.Species("empty_traps", mobile=False)  # NEW: an explicit species tracking how many empty trap sites are left, immobile since it's just a site count not something physically diffusing

my_model.species = [mobile_H, trapped_H, empty_traps]

material = F.Material(D_0=1, E_D=0)  # just redefining the same thing again, no actual difference from above

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)
my_model.initial_conditions = [F.InitialConcentration(value=2, species=empty_traps, volume=vol)]  # NEW: sets empty_traps=2 everywhere at t=0, like giving u_n a nonzero starting value instead of defaulting to 0 - this is our "n" (trap density) from eg_4.py, just represented as its own species instead of a fixed constant

```{note}
Here we set the empty traps as immobile, but they could diffuse in the material too by setting ``mobile=True``.
```

In [ ]:
left_surf = F.SurfaceSubdomain1D(id=1, x=0)
right_surf = F.SurfaceSubdomain1D(id=2, x=1)

my_model.subdomains = [vol, left_surf, right_surf]

my_model.reactions = [
    F.Reaction(
        reactant=[mobile_H, empty_traps],  # this is exactly our reaction = -k*cm*(n-ct)+p*ct, just written as two separate reactants instead of us writing (n-ct) ourselves - festim handles depleting empty_traps as trapped_H grows automatically
        product=[trapped_H],
        k_0=0.01,
        E_k=0,
        p_0=0.1,
        E_p=0,
        volume=vol,
    ),
]

my_model.boundary_conditions = [
    F.FixedConcentrationBC(left_surf, value=10, species=mobile_H),  # only mobile_H gets a BC - same as our cm being the only species with BCs in eg_4.py, trapped/empty traps have none since they're purely reaction driven
    F.FixedConcentrationBC(right_surf, value=0, species=mobile_H),
]

my_model.temperature = 300

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=50)

my_model.settings.stepsize = F.Stepsize(1)

my_model.initialise()
my_model.run()

In [ ]:
for species in my_model.species:
    plot_profile(species, label=species.name)

plt.xlabel("Position")
plt.ylabel("Concentration")
plt.legend()
plt.show()

### Implicit trapping sites

As showed in the [Species](species.ipynb) tutorial, in this case, empty traps can be defined as an implicit species, and their concentration is defined as:

$$
n_\mathrm{empty} = n_\mathrm{total} - c_\mathrm{t}
$$

In [ ]:
my_model = F.HydrogenTransportProblem()
my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))

mobile_H = F.Species("H")
trapped_H = F.Species("H_trapped", mobile=False)
empty_traps = F.ImplicitSpecies(n=2, others=[trapped_H])  # NEW: instead of solving for empty_traps as its own DOF, this just computes it algebraically as n - trapped_H, exactly like our (n - ct) term written directly into reaction. saves us a whole extra field in the mixed space

my_model.species = [mobile_H, trapped_H]  # note empty_traps is NOT listed here - it's implicit, it never actually gets its own DOFs/mixed space slot, unlike the explicit version above

left_surf = F.SurfaceSubdomain1D(id=1, x=0)
right_surf = F.SurfaceSubdomain1D(id=2, x=1)

material = F.Material(D_0=1, E_D=0)

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)

my_model.subdomains = [vol, left_surf, right_surf]

my_model.reactions = [
    F.Reaction(
        reactant=[mobile_H, empty_traps],  # same reaction as the explicit version, just empty_traps is now the implicit (n - trapped_H) form
        product=[trapped_H],
        k_0=0.01,
        E_k=0,
        p_0=0.1,
        E_p=0,
        volume=vol,
    ),
]

my_model.boundary_conditions = [
    F.FixedConcentrationBC(left_surf, value=10, species=mobile_H),
    F.FixedConcentrationBC(right_surf, value=0, species=mobile_H),
]

my_model.temperature = 300

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=50)

my_model.settings.stepsize = F.Stepsize(1)

my_model.initialise()
my_model.run()

In [ ]:
for species in my_model.species:
    plot_profile(species, label=species.name)

plt.xlabel("Position")
plt.ylabel("Concentration")
plt.legend()
plt.show()

### Convenience class for trapping

For simple cases (ie. one mobile species, one hydrogen per trap), FESTIM has a convenience class ``Trap`` to simplify the inputs.

With this class, all that is needed is defining the trapping rates, the mobile species, the total number of traps. It provides a slightly more compact way of defining traps, at the cost of flexibility.

In [ ]:
my_model = F.HydrogenTransportProblem()
my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))

material = F.Material(D_0=1, E_D=0)
left_surf = F.SurfaceSubdomain1D(id=1, x=0)
right_surf = F.SurfaceSubdomain1D(id=2, x=1)
vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)
my_model.subdomains = [vol, left_surf, right_surf]

mobile_H = F.Species("H")
my_model.species = [mobile_H]  # only mobile_H listed here - Trap below builds the trapped species + empty_traps + the reaction all for us, we never see them directly

my_model.traps = [
    F.Trap(
        mobile_species=mobile_H,
        k_0=0.01,
        E_k=0,
        p_0=0.1,
        E_p=0,
        volume=vol,
        n=2,  # number of traps
        name="Trap1",
    )
]
# this one F.Trap call replaces everything we did by hand above (trapped_H species + empty_traps species/ImplicitSpecies + the F.Reaction) - shortcut for the common single-trap, one-H-per-trap case, less flexible but way less typing

my_model.boundary_conditions = [
    F.FixedConcentrationBC(left_surf, value=10, species=mobile_H),
    F.FixedConcentrationBC(right_surf, value=0, species=mobile_H),
]

my_model.temperature = 300
my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=50)
my_model.settings.stepsize = F.Stepsize(1)

my_model.initialise()
my_model.run()

In [ ]:
for species in my_model.species:
    plot_profile(species, label=species.name)

plt.xlabel("Position")
plt.ylabel("Concentration")
plt.legend()
plt.show()

### Multi-occupancy trapping

So far we've assumed that a trap could only accomodate one H atom.
However, it is possible to simulate the fact that one trap can hold several atoms.
This is called the multi-occupancy trap model.

Let's consider a trap with 3 occupancy levels. We have the following reaction scheme:

```{math}
\ce{H + [ ] <-->[k][p] [1H]}
```

```{math}
\ce{H + [1H] <-->[k][p] [2H]}
```

```{math}
\ce{H + [2H] <-->[k][p] [3H]}
```

The underlying governing equations are:

\begin{align}
    \frac{\partial c_\mathrm{m}}{\partial t} &= \nabla \cdot (D \nabla c_\mathrm{m}) - k c_\mathrm{m} n_\mathrm{empty} + p c_\mathrm{t,1} - k c_\mathrm{m} c_\mathrm{t,1} + p c_\mathrm{t,2} - k c_\mathrm{m} c_\mathrm{t,2} + p c_\mathrm{t,3}\\
    \frac{\partial c_\mathrm{t,1}}{\partial t} &= k c_\mathrm{m} n_\mathrm{empty} - p c_\mathrm{t,1} - k c_\mathrm{m} c_\mathrm{t,1} + p c_\mathrm{t,2}\\
    \frac{\partial c_\mathrm{t,2}}{\partial t} &= k c_\mathrm{m} c_\mathrm{t,1} - p c_\mathrm{t,2} - k c_\mathrm{m} c_\mathrm{t,2} + p c_\mathrm{t,3}\\
    \frac{\partial c_\mathrm{t,2}}{\partial t} &= k c_\mathrm{m} c_\mathrm{t,2} - p c_\mathrm{t,3}\\
\end{align}

with:

\begin{equation}
    n_\mathrm{empty} = n_\mathrm{total} - \sum c_\mathrm{t, i}
\end{equation}

In [ ]:
my_model = F.HydrogenTransportProblem()
my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))


mobile_H = F.Species("H")
trapped_1H = F.Species("1H_trapped", mobile=False)  # trap holding 1 H atom
trapped_2H = F.Species("2H_trapped", mobile=False)  # trap holding 2 H atoms
trapped_3H = F.Species("3H_trapped", mobile=False)  # trap holding 3 H atoms - three separate occupancy levels, each its own species
empty_traps = F.ImplicitSpecies(n=2, others=[trapped_1H, trapped_2H, trapped_3H])  # NEW: n minus ALL three occupied levels this time, not just one - so empty_traps = n - 1H - 2H - 3H

my_model.species = [mobile_H, trapped_1H, trapped_2H, trapped_3H]

left_surf = F.SurfaceSubdomain1D(id=1, x=0)
right_surf = F.SurfaceSubdomain1D(id=2, x=1)

material = F.Material(D_0=1, E_D=0)

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)

my_model.subdomains = [vol, left_surf, right_surf]

my_model.reactions = [
    F.Reaction(
        reactant=[mobile_H, empty_traps],  # step 1: H fills an empty trap -> 1H occupied
        product=[trapped_1H],
        k_0=0.01,
        E_k=0,
        p_0=0.1,
        E_p=0,
        volume=vol,
    ),
    F.Reaction(
        reactant=[mobile_H, trapped_1H],  # step 2: another H fills an already-1H trap -> 2H occupied. note the reactant here is trapped_1H, not empty_traps - this is the ladder mechanism
        product=[trapped_2H],
        k_0=0.02,  # slightly faster rate for this step, just picked to show they don't have to match
        E_k=0,
        p_0=0.1,
        E_p=0,
        volume=vol,
    ),
    F.Reaction(
        reactant=[mobile_H, trapped_2H],  # step 3: same idea, 2H -> 3H
        product=[trapped_3H],
        k_0=0.03,
        E_k=0,
        p_0=0.1,
        E_p=0,
        volume=vol,
    ),
]

my_model.boundary_conditions = [
    F.FixedConcentrationBC(left_surf, value=10, species=mobile_H),
    F.FixedConcentrationBC(right_surf, value=0, species=mobile_H),
]

my_model.temperature = 300

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=50)

my_model.settings.stepsize = F.Stepsize(1)

my_model.initialise()
my_model.run()

In [ ]:
for species in my_model.species:
    plot_profile(species, label=species.name)

plt.xlabel("Position")
plt.ylabel("Concentration")
plt.legend()
plt.show()

### Two species, one trap

Now we consider a unique trapping site that can react with two different mobile species H and D:

```{math}
\ce{H + [\ ] <-->[k][p] [H]}
```

```{math}
\ce{D + [\ ] <-->[k][p] [D]}
```

The governing equations for this problem are:


\begin{align}
    \frac{\partial c_\mathrm{m, H}}{\partial t} &= \nabla \cdot (D \nabla c_\mathrm{m, H}) - k c_\mathrm{m, H} n_\mathrm{empty} + p c_\mathrm{t, H}\\
    \frac{\partial c_\mathrm{m, D}}{\partial t} &= \nabla \cdot (D \nabla c_\mathrm{m, D}) - k c_\mathrm{m, D} n_\mathrm{empty} + p c_\mathrm{t, D}\\
    \frac{\partial c_\mathrm{t, H}}{\partial t} &= k c_\mathrm{m,H} n_\mathrm{empty} - p c_\mathrm{t,H}\\
    \frac{\partial c_\mathrm{t,D}}{\partial t} &= k c_\mathrm{m,D} n_\mathrm{empty} - p c_\mathrm{t,D}
\end{align}

with

\begin{equation}
    n_\mathrm{empty} = n_\mathrm{total} - \sum c_\mathrm{t, i}
\end{equation}


$c_\mathrm{m,i}$ is the concentration of species i and $c_\mathrm{t,i}$ is the concentration of species [i].

In [ ]:
my_model = F.HydrogenTransportProblem()
my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))


mobile_H = F.Species("H")
mobile_T = F.Species("T")  # a second, independent mobile species (like our earlier cm/c_o pair) - this one is tritium sharing the domain with H
trapped_H = F.Species("H_trapped", mobile=False)
trapped_T = F.Species("T_trapped", mobile=False)
empty_traps = F.ImplicitSpecies(n=2, others=[trapped_H, trapped_T])  # NEW: same trap pool shared by BOTH species - empty_traps = n - trapped_H - trapped_T, so H and T are competing for the same limited sites

my_model.species = [mobile_H, mobile_T, trapped_H, trapped_T]


left_surf = F.SurfaceSubdomain1D(id=1, x=0)
right_surf = F.SurfaceSubdomain1D(id=2, x=1)

material = F.Material(D_0=1, E_D=0)

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)

my_model.subdomains = [vol, left_surf, right_surf]

my_model.reactions = [
    F.Reaction(
        reactant=[mobile_H, empty_traps],  # H trapping - draws from the shared pool
        product=[trapped_H],
        k_0=0.1,
        E_k=0,
        p_0=0.001,
        E_p=0,
        volume=vol,
    ),
    F.Reaction(
        reactant=[mobile_T, empty_traps],  # T trapping - draws from the SAME shared pool, so whichever species gets there first uses up sites the other one would've used
        product=[trapped_T],
        k_0=0.1,
        E_k=0,
        p_0=0.001,
        E_p=0,
        volume=vol,
    ),
]

my_model.boundary_conditions = [
    F.FixedConcentrationBC(left_surf, value=10, species=mobile_H),
    F.FixedConcentrationBC(right_surf, value=0, species=mobile_H),
    F.FixedConcentrationBC(left_surf, value=0, species=mobile_T),  # note H and T come in from opposite ends (H high on the left, T high on the right) so they meet in the middle
    F.FixedConcentrationBC(right_surf, value=5, species=mobile_T),
]

my_model.temperature = 300

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=20)

my_model.settings.stepsize = F.Stepsize(1)

my_model.initialise()
my_model.run()

In [ ]:
for species in my_model.species:
    if "T" in species.name:
        color = "tab:green"  # T species get green
    else:
        color = "tab:blue"   # H species get blue - just picking apart the name string to colour-code, nothing fancy
    if "trapped" in species.name:
        linestyle = "--"  # dashed for trapped species
    else:
        linestyle = "-"   # solid for mobile species - so we can tell all four lines apart at a glance
    plot_profile(species, label=species.name, color=color, linestyle=linestyle)

plt.xlabel("Position")
plt.ylabel("Concentration")
plt.legend()
plt.show()

## Isotope swapping

``Reaction`` objects can have multiple products. This is useful to simulate things like isotope swapping where a mobile species reacts with a trapped species, and swap their positions:

```{math}
\ce{H + [T] <-->[k_\mathrm{swap}][k_\mathrm{swap}] T + [H]}
```

In this example we consider the above reaction, with an initial condition for the trapped T concentration and mobile H diffusing in the domain, slowly swapping trapped T.

The governing equations for this problem are:


\begin{align}
    \frac{\partial c_\mathrm{m, H}}{\partial t} &= \nabla \cdot (D \nabla c_\mathrm{m, H}) - k_\mathrm{swap} (c_\mathrm{m, H} \ c_\mathrm{t, D} - c_\mathrm{m, D} \ c_\mathrm{t, H})\\
    \frac{\partial c_\mathrm{m, T}}{\partial t} &= \nabla \cdot (D \nabla c_\mathrm{m, T}) + k_\mathrm{swap} (c_\mathrm{m, H} \ c_\mathrm{t, D} - c_\mathrm{m, D} \ c_\mathrm{t, H})\\
    \frac{\partial c_\mathrm{t, H}}{\partial t} &= k_\mathrm{swap} (c_\mathrm{m, H} \ c_\mathrm{t, D} - c_\mathrm{m, D} \ c_\mathrm{t, H})\\
    \frac{\partial c_\mathrm{t, T}}{\partial t} &=  - k_\mathrm{swap} (c_\mathrm{m, H} \ c_\mathrm{t, D} - c_\mathrm{m, D} \ c_\mathrm{t, H})
\end{align}

In [ ]:
my_model = F.HydrogenTransportProblem()
my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))

material = F.Material(D_0=1, E_D=0)

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)

mobile_H = F.Species("H")
mobile_T = F.Species("T")
trapped_H = F.Species("H_trapped", mobile=False)
trapped_T = F.Species("T_trapped", mobile=False)

my_model.species = [mobile_H, mobile_T, trapped_H, trapped_T]

my_model.initial_conditions = [
    F.InitialConcentration(value=2, species=trapped_T, volume=vol),  # starts with traps already full of T everywhere, so H coming in from the left has something to swap with
]

left_surf = F.SurfaceSubdomain1D(id=1, x=0)
right_surf = F.SurfaceSubdomain1D(id=2, x=1)



my_model.subdomains = [vol, left_surf, right_surf]

my_model.initial_conditions = [
    F.InitialConcentration(value=2, species=trapped_T, volume=vol),  # same as above, just set again - no functional difference
]

my_model.reactions = [
    F.Reaction(
        reactant=[mobile_H, trapped_T],   # NEW: mobile H reacts with trapped T (not empty traps this time)
        product=[mobile_T, trapped_H],    # NEW: TWO products - H takes T's spot in the trap, and the displaced T becomes mobile. this is the actual "swap" - nothing about total trap occupancy changes, just who's in the trap vs who's free
        k_0=0.005,
        E_k=0,
        p_0=0.005,  # same rate forward/backward here, so equally likely to swap either direction
        E_p=0,
        volume=vol,
    ),
]

my_model.boundary_conditions = [
    F.FixedConcentrationBC(left_surf, value=10, species=mobile_H),
    F.FixedConcentrationBC(right_surf, value=0, species=mobile_H),
]
# note there's no BC on mobile_T - it only appears in the domain because it gets displaced out of the traps by H, not from a boundary source

my_model.temperature = 300

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=10)

my_model.settings.stepsize = F.Stepsize(1)

my_model.initialise()
my_model.run()

In [ ]:
# same colour/style-coding as the previous plot - green for T, blue for H, dashed for trapped
for species in my_model.species:
    if "T" in species.name:
        color = "tab:green"
    else:
        color = "tab:blue"
    if "trapped" in species.name:
        linestyle = "--"
    else:
        linestyle = "-"
    plot_profile(species, label=species.name, color=color, linestyle=linestyle)

plt.xlabel("Position")
plt.ylabel("Concentration")
plt.legend()
plt.show()

## Annihilation

``Reaction`` objects can also have no products at all, simulating annihiliation reactions.

### Radioactive decay

When dealing with radioactive species, this type of reaction can be used to simulate radioactive decay. The rate of the reaction being equal to the decay constant.

Here we consider a species A decaying with a decay constant $\lambda$.


```{math}
\ce{A ->[\lambda] \emptyset}
```

The governing equation is:

\begin{equation}
    \frac{\partial c_\mathrm{A}}{\partial t} = \nabla \cdot (D \nabla c_\mathrm{A}) - \lambda c_\mathrm{A}
\end{equation}

In [ ]:
my_model = F.HydrogenTransportProblem()
my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))

material = F.Material(D_0=1, E_D=0)
vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)
left_surf = F.SurfaceSubdomain1D(id=1, x=0)
right_surf = F.SurfaceSubdomain1D(id=2, x=1)

my_model.subdomains = [vol, left_surf, right_surf]

A = F.Species("A")

my_model.species = [A]
left_surf = F.SurfaceSubdomain1D(id=1, x=0)  # duplicated from above, no functional difference
right_surf = F.SurfaceSubdomain1D(id=2, x=1)

material = F.Material(D_0=1, E_D=0)  # duplicated again

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)

my_model.subdomains = [vol, left_surf, right_surf]  # set again

my_model.initial_conditions = [F.InitialConcentration(value=1, species=A, volume=vol)]  # starts with A=1 everywhere, nothing coming in from a boundary this time - purely watching it decay away

my_model.reactions = [
    F.Reaction(reactant=[A], k_0=1, E_k=0, volume=vol),  # NEW: no product= at all! this is exactly the tritium decay term we worked out by hand (F += lam*u*v*dx) - k_0 here IS the decay constant lambda, A just disappears at this rate, nothing replaces it
]

my_model.temperature = 300

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=1)

my_model.settings.stepsize = F.Stepsize(1)

my_model.initialise()
my_model.run()

In [ ]:
x = my_model.mesh.mesh.geometry.x[:, 0]  # NEW plotting approach - reading raw mesh point coordinates directly since there's only one species, no mixed space to collapse out of
c = A.post_processing_solution.x.array[:]  # same idea as our .x.array reads, just via species.post_processing_solution instead of us manually tracking cm_pp etc

plt.plot(x, c, label=A.name)
plt.axhline(2, linestyle="--", color="green", label="Initial concentration")  # NOTE: this says 2 but the initial_conditions above set value=1, so this reference line doesn't actually match - looks like a copy-paste mismatch in the original notebook

plt.xlabel("Position")
plt.ylabel("Concentration")
plt.ylim(bottom=0)
plt.legend()
plt.show()

### Vacancy-interstitial annihilation

Another use case of this type of reaction is metal interstitial atoms recombining with vacancies (both forming a Frenkel pair) and annihilate.

```{math}
\ce{I + V ->[k] \emptyset}
```

In this example, we define two mobile species I and V, both diffusing from one side of the 1D domain and annihilating.

The governing equations are:

\begin{align}
    \frac{\partial c_\mathrm{I}}{\partial t} &= \nabla \cdot (D \nabla c_\mathrm{I}) - k c_\mathrm{I} c_\mathrm{V}\\
    \frac{\partial c_\mathrm{V}}{\partial t} &= \nabla \cdot (D \nabla c_\mathrm{V}) - k c_\mathrm{I} c_\mathrm{V}
\end{align}

In [ ]:
my_model = F.HydrogenTransportProblem()
my_model.mesh = F.Mesh1D(np.linspace(0, 1, 100))

I = F.Species("I")
V = F.Species("V")

my_model.species = [I, V]

left_surf = F.SurfaceSubdomain1D(id=1, x=0)
right_surf = F.SurfaceSubdomain1D(id=2, x=1)

material = F.Material(D_0=1, E_D=0)

vol = F.VolumeSubdomain1D(id=1, borders=[0, 1], material=material)

my_model.subdomains = [vol, left_surf, right_surf]

my_model.reactions = [
    F.Reaction(reactant=[I, V], k_0=1000, E_k=0, volume=vol),  # NEW: two reactants, no product - true annihilation, both I and V just vanish together, nothing left over unlike our h2o case. k_0=1000 is a big rate constant so this happens almost instantly wherever I and V overlap
]
my_model.boundary_conditions = [
    F.FixedConcentrationBC(left_surf, value=10, species=I),
    F.FixedConcentrationBC(right_surf, value=0, species=I),
    F.FixedConcentrationBC(left_surf, value=0, species=V),  # I comes in from the left, V comes in from the right, so they diffuse towards each other and annihilate somewhere in the middle
    F.FixedConcentrationBC(right_surf, value=5, species=V),
]
my_model.temperature = 300

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, transient=False)  # NEW: transient=False - steady state solve instead of time-stepping, so no final_time/stepsize needed here, it just solves for the final equilibrium profile directly

my_model.initialise()
my_model.run()

In [ ]:
for species in my_model.species:
    l, = plot_profile(species, label=species.name)  # plot_profile returns a list from plt.plot, unpacking the single Line2D object out of it as l
    plt.fill_between(
        l.get_data()[0],   # NEW: shading the area under each curve - x data pulled back off the line object
        0,
        l.get_data()[1],   # y data off the same line
        alpha=0.2,         # translucent so overlapping regions (where I and V both existed before annihilating) are still visible
        color=l.get_color(),  # matches the fill colour to whatever colour matplotlib auto-assigned the line
    )


plt.xlabel("Position")
plt.ylabel("Concentration")
plt.legend()
plt.show()